[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/01_Multimodal_Foundations/01_what_is_multimodal/01_what_is_multimodal.ipynb)

# 01. What is Multimodal Learning?

**You already know:** Self-attention, Multi-head attention, Cross-attention, Transformers

**This notebook covers:**
- What "multimodal" means and why it matters
- The landscape of multimodal models (CLIP, LLaVA, Flamingo, etc.)
- How different modalities are represented
- The key challenge: aligning different representation spaces
- **Hands-on:** Build a Mini-CLIP model from scratch

**Runtime:** ~5 minutes on CPU | ~2 minutes on Colab GPU

---

In [ ]:
# ============================================================
#  Google Colab Setup — Run this cell FIRST
# ============================================================
import os, sys

try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab detected — setting up environment...")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    # Step 1: Clone the repository
    if not os.path.exists(REPO_DIR):
        print("📥 Cloning repository...")
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        print("✅ Repository already cloned")

    # Step 2: Install dependencies (Colab-safe subset)
    print("📦 Installing dependencies...")
    !pip install -q torch torchvision torchaudio
    !pip install -q transformers datasets accelerate peft
    !pip install -q matplotlib seaborn numpy pandas scikit-learn tqdm
    !pip install -q einops timm sentencepiece tokenizers safetensors
    !pip install -q open-clip-torch gradio onnx onnxruntime
    !pip install -q ipywidgets pillow

    # Step 3: Set working directory
    MODULE_DIR = f"{REPO_DIR}/01_Multimodal_Foundations/01_what_is_multimodal"
    os.chdir(MODULE_DIR)
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)

    # Step 4: Add project root to Python path
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"\n✅ Colab setup complete!")
    print(f"   Working directory: {os.getcwd()}")

    # Show GPU info if available
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"   GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    else:
        print("   Device: CPU (all notebooks work fine on CPU)")
else:
    os.makedirs("../assets", exist_ok=True)
    print("Running locally — all set!")

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np

# Import project utilities (visualization helpers)
try:
    from utils.visualization import *
    from utils.helpers import count_parameters
    set_style()
    print("✅ Utilities loaded successfully")
except ImportError as e:
    print(f"⚠️  Could not import utils ({e})")
    print("   Defining inline fallbacks — notebook will still work fine.")

    import seaborn as sns

    def set_style():
        plt.rcParams.update({
            'figure.figsize': (12, 6), 'figure.dpi': 100,
            'font.size': 11, 'axes.titlesize': 14, 'axes.labelsize': 12,
            'axes.grid': True, 'grid.alpha': 0.3, 'figure.facecolor': 'white',
        })
        sns.set_palette("husl")

    def draw_architecture_block(ax, x, y, w, h, label, color='#4ECDC4', fontsize=10):
        box = FancyBboxPatch((x - w/2, y - h/2), w, h,
            boxstyle="round,pad=0.1", facecolor=color, edgecolor='#2C3E50',
            linewidth=1.5, alpha=0.85)
        ax.add_patch(box)
        ax.text(x, y, label, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color='white')
        return box

    def draw_arrow(ax, start, end, color='#2C3E50', style='->', lw=1.5):
        ax.annotate('', xy=end, xytext=start,
                    arrowprops=dict(arrowstyle=style, color=color, lw=lw))

    def plot_multimodal_overview():
        set_style()
        fig, ax = plt.subplots(1, 1, figsize=(14, 8))
        ax.set_xlim(0, 14); ax.set_ylim(0, 8); ax.axis('off')
        ax.set_title('Multimodal Architecture Overview', fontsize=18, fontweight='bold', pad=20)
        colors = {'image': '#E74C3C', 'text': '#3498DB', 'audio': '#2ECC71',
                  'fusion': '#9B59B6', 'output': '#F39C12', 'encoder': '#1ABC9C'}
        draw_architecture_block(ax, 2, 6.5, 2.5, 1, 'Image\n(pixels)', colors['image'])
        draw_architecture_block(ax, 7, 6.5, 2.5, 1, 'Text\n(tokens)', colors['text'])
        draw_architecture_block(ax, 12, 6.5, 2.5, 1, 'Audio\n(spectrogram)', colors['audio'])
        draw_architecture_block(ax, 2, 4.5, 2.5, 1, 'Vision Encoder\n(ViT / CNN)', colors['encoder'])
        draw_architecture_block(ax, 7, 4.5, 2.5, 1, 'Text Encoder\n(BERT / GPT)', colors['encoder'])
        draw_architecture_block(ax, 12, 4.5, 2.5, 1, 'Audio Encoder\n(Whisper)', colors['encoder'])
        for x in [2, 7, 12]:
            draw_arrow(ax, (x, 6.0), (x, 5.1))
        draw_architecture_block(ax, 7, 2.5, 8, 1.2,
            'Fusion Layer\n(Cross-Attention / Concatenation / Gating)', colors['fusion'], fontsize=12)
        for x in [2, 7, 12]:
            draw_arrow(ax, (x, 4.0), (x if x == 7 else (4.5 if x == 2 else 9.5), 3.2))
        draw_architecture_block(ax, 7, 0.8, 4, 0.9,
            'Task Output (Classification / Generation)', colors['output'], fontsize=10)
        draw_arrow(ax, (7, 1.9), (7, 1.35))
        plt.tight_layout()
        return fig

    def plot_attention_heatmap(attention_weights, x_labels=None, y_labels=None, title='Attention Weights'):
        set_style()
        if isinstance(attention_weights, torch.Tensor):
            attention_weights = attention_weights.detach().cpu().numpy()
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(attention_weights, annot=True, fmt='.3f', cmap='YlOrRd',
                    xticklabels=x_labels, yticklabels=y_labels,
                    ax=ax, cbar_kws={'label': 'Attention Weight'})
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.set_xlabel('Keys'); ax.set_ylabel('Queries')
        plt.tight_layout()
        return fig

    def count_parameters(model, print_table=True):
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        frozen = total - trainable
        if print_table:
            print(f"{'='*50}")
            print(f"{'Parameter Summary':^50}")
            print(f"{'='*50}")
            print(f"  Total parameters:     {total:>12,}")
            print(f"  Trainable parameters: {trainable:>12,}")
            print(f"  Frozen parameters:    {frozen:>12,}")
            print(f"  Trainable %:          {trainable/total*100:>11.2f}%")
            print(f"{'='*50}")
        return {'total': total, 'trainable': trainable, 'frozen': frozen}

    set_style()
    print("✅ Inline utilities ready")

print(f"\nPyTorch version: {torch.__version__}")
print(f"Device: {'cuda (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")

## 1. The Multimodal Landscape

**Single-modal:** One input type (text OR image OR audio)  
**Multi-modal:** Multiple input types working together

| Model | Year | Modalities | What it does | Open Source? |
|-------|------|-----------|-------|------------|
| **CLIP** | 2021 | Image + Text | Matches images to text descriptions | ✅ Yes |
| **LLaVA** | 2023 | Image + Text | Answers questions about images | ✅ Yes |
| **Flamingo** | 2022 | Image + Text | Few-shot visual reasoning | ❌ No |
| **Whisper** | 2022 | Audio → Text | Speech to text | ✅ Yes |
| **ImageBind** | 2023 | 6 modalities | Unified embedding space | ✅ Yes |
| **GPT-4V** | 2023 | Image + Text | General multimodal reasoning | ❌ No |
| **Gemini** | 2023 | Native multi-modal | Reasoning across modalities | ❌ No |

> **Key insight:** Almost all of these share the same recipe — **separate encoders + shared embedding space + alignment training**. We'll build this pattern from scratch below.

In [ ]:
# Visualize the multimodal architecture overview
fig = plot_multimodal_overview()
plt.savefig('../assets/multimodal_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to assets/multimodal_overview.png")

## 2. How Different Modalities Are Represented

The core challenge: **every modality has a different native format.**
We need to convert them all into a **common representation** (embedding vectors).

| Modality | Raw Format | Encoder | Output |
|----------|-----------|---------|--------|
| **Image** | Pixels $\in \mathbb{R}^{H \times W \times 3}$ | ViT / CNN | $\mathbf{h} \in \mathbb{R}^D$ |
| **Text** | Token IDs $\in \mathbb{Z}^T$ | BERT / GPT | $\mathbf{h} \in \mathbb{R}^D$ |
| **Audio** | Waveform $\in \mathbb{R}^L$ | Whisper / wav2vec | $\mathbf{h} \in \mathbb{R}^D$ |

> Notice: all encoders output **the same shape** $\mathbf{h} \in \mathbb{R}^D$. That's the key to multimodal fusion!

In [ ]:
# Visualize: How each modality becomes a vector
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Image → Patches → Embeddings
ax = axes[0]
ax.set_title('Image → Embeddings', fontsize=14, fontweight='bold')
img = np.random.rand(8, 8, 3) * 0.5 + 0.3
ax.imshow(img, extent=[0, 4, 4, 8])

# Show patch grid
for i in range(0, 5, 2):
    ax.axhline(y=4+i, xmin=0, xmax=0.4, color='white', lw=2)
    ax.axvline(x=i*0.5, ymin=0.5, ymax=1.0, color='white', lw=2)

# Show embedding vectors
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']
for i, c in enumerate(colors):
    vec = np.random.randn(1, 10) * 0.5
    ax.barh(3 - i * 0.7, vec[0], height=0.5, color=c, alpha=0.7, left=0)
    ax.text(-2.5, 3 - i * 0.7, f'patch {i+1}', fontsize=9, va='center')

ax.set_xlim(-3, 5)
ax.set_ylim(-0.5, 8.5)
ax.arrow(2, 3.8, 0, -0.5, head_width=0.2, color='black')
ax.text(2, 4.2, 'ViT / CNN', ha='center', fontsize=10, fontweight='bold')
ax.axis('off')

# Text → Tokens → Embeddings
ax = axes[1]
ax.set_title('Text → Embeddings', fontsize=14, fontweight='bold')
tokens = ['[CLS]', 'a', 'cat', 'sitting', '[SEP]']
for i, tok in enumerate(tokens):
    ax.add_patch(FancyBboxPatch((i*1.5+0.5, 6), 1.2, 0.8, 
                                boxstyle='round,pad=0.1', facecolor='#3498DB', alpha=0.7))
    ax.text(i*1.5+1.1, 6.4, tok, ha='center', va='center', fontsize=9, color='white', fontweight='bold')

for i in range(5):
    vec = np.random.randn(1, 10) * 0.5
    ax.barh(4 - i * 0.7, vec[0], height=0.5, color='#3498DB', alpha=0.6, left=0.5)
    ax.text(-1, 4 - i * 0.7, tokens[i], fontsize=9, va='center')

ax.arrow(4, 5.8, 0, -0.5, head_width=0.2, color='black')
ax.text(4, 6.2, 'Tokenizer + Encoder', ha='center', fontsize=10, fontweight='bold')
ax.set_xlim(-2, 9)
ax.set_ylim(0.5, 8)
ax.axis('off')

# Audio → Spectrogram → Embeddings
ax = axes[2]
ax.set_title('Audio → Embeddings', fontsize=14, fontweight='bold')
spec = np.random.rand(20, 40) ** 2
ax.imshow(spec, aspect='auto', cmap='magma', extent=[0.5, 7.5, 4.5, 7.5])
ax.text(4, 7.8, 'Mel Spectrogram', ha='center', fontsize=10)

for i in range(4):
    vec = np.random.randn(1, 10) * 0.5
    ax.barh(3.5 - i * 0.7, vec[0], height=0.5, color='#2ECC71', alpha=0.6, left=0.5)
    ax.text(-1, 3.5 - i * 0.7, f'frame {i+1}', fontsize=9, va='center')

ax.arrow(4, 4.3, 0, -0.3, head_width=0.2, color='black')
ax.text(4, 4.7, 'Whisper / wav2vec', ha='center', fontsize=10, fontweight='bold')
ax.set_xlim(-2, 9)
ax.set_ylim(0.5, 8.5)
ax.axis('off')

plt.tight_layout()
plt.savefig('../assets/modality_representations.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. The Alignment Problem (Core Challenge)

**Problem:** A cat image and the text "a photo of a cat" should map to **nearby** points in embedding space.

Mathematically, we want the **cosine similarity** between matching pairs to be high:

$$\text{sim}(\mathbf{v}_{\text{cat}}, \mathbf{t}_{\text{"a cat"}}) = \frac{\mathbf{v}_{\text{cat}} \cdot \mathbf{t}_{\text{"a cat"}}}{\|\mathbf{v}_{\text{cat}}\| \cdot \|\mathbf{t}_{\text{"a cat"}}\|} \to 1$$

And similarity between non-matching pairs to be low:

$$\text{sim}(\mathbf{v}_{\text{cat}}, \mathbf{t}_{\text{"a dog"}}) \to 0$$

**Before training:** All similarities are random (~0).  
**After training:** Matching pairs cluster together!

### Cosine Similarity — Numerical Walkthrough

Let's compute similarity by hand with 4-dimensional vectors:

$$\mathbf{v} = [0.3,\; 0.8,\; 0.1,\; 0.5], \quad \mathbf{t} = [0.4,\; 0.7,\; 0.2,\; 0.6]$$

**Step 1 — Dot product:**

$$\mathbf{v} \cdot \mathbf{t} = (0.3)(0.4) + (0.8)(0.7) + (0.1)(0.2) + (0.5)(0.6) = 0.12 + 0.56 + 0.02 + 0.30 = 1.00$$

**Step 2 — L2 norms:**

$$\|\mathbf{v}\| = \sqrt{0.09 + 0.64 + 0.01 + 0.25} = \sqrt{0.99} \approx 0.995$$

$$\|\mathbf{t}\| = \sqrt{0.16 + 0.49 + 0.04 + 0.36} = \sqrt{1.05} \approx 1.025$$

**Step 3 — Cosine similarity:**

$$\text{sim}(\mathbf{v}, \mathbf{t}) = \frac{1.00}{0.995 \times 1.025} \approx \frac{1.00}{1.020} \approx 0.980$$

A score of **0.98** means these vectors point in nearly the same direction — exactly what we want for a matching image-text pair after alignment training.

### L2 Normalization — Why It Matters

Before computing cosine similarity, CLIP **L2-normalizes** both embedding vectors:

$$\hat{\mathbf{v}} = \frac{\mathbf{v}}{\|\mathbf{v}\|_2}, \quad \hat{\mathbf{t}} = \frac{\mathbf{t}}{\|\mathbf{t}\|_2}$$

**Three critical reasons:**

1. **Dot product = cosine similarity:** After normalization, $\hat{\mathbf{v}} \cdot \hat{\mathbf{t}} = \cos(\theta)$ — no need to compute norms separately at inference time
2. **Bounded output:** Cosine similarity is always in $[-1, 1]$, preventing any single embedding dimension from dominating the score
3. **Training stability:** Without normalization, the model can cheat by scaling embeddings larger instead of learning meaningful directions — L2 norm forces the model to learn **direction** (semantic content), not magnitude

This is why our `MiniCLIP` applies `emb / emb.norm(dim=-1, keepdim=True)` before computing the similarity matrix.

In [ ]:
# Demonstrate the alignment problem visually
np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# BEFORE alignment
ax = axes[0]
ax.set_title('BEFORE Alignment Training', fontsize=14, fontweight='bold', color='#E74C3C')

# Image embeddings (clustered in one region)
img_pts = np.random.randn(5, 2) * 0.8 + np.array([3, 3])
txt_pts = np.random.randn(5, 2) * 0.8 + np.array([-2, -2])

labels = ['cat', 'dog', 'car', 'tree', 'house']
ax.scatter(img_pts[:, 0], img_pts[:, 1], s=150, c='#E74C3C', marker='s', 
           label='Image embeddings', zorder=5, edgecolors='white', linewidths=1.5)
ax.scatter(txt_pts[:, 0], txt_pts[:, 1], s=150, c='#3498DB', marker='o', 
           label='Text embeddings', zorder=5, edgecolors='white', linewidths=1.5)

for i, label in enumerate(labels):
    ax.annotate(f'img:{label}', img_pts[i], fontsize=9, 
                xytext=(5, 5), textcoords='offset points')
    ax.annotate(f'txt:{label}', txt_pts[i], fontsize=9, 
                xytext=(5, 5), textcoords='offset points')

ax.legend(fontsize=11)
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')

# AFTER alignment (matching pairs are close)
ax = axes[1]
ax.set_title('AFTER Alignment Training (e.g. CLIP)', fontsize=14, fontweight='bold', color='#2ECC71')

centers = np.array([[-2, 3], [2, 3], [3, -1], [-3, -1], [0, -3]])
img_pts = centers + np.random.randn(5, 2) * 0.15
txt_pts = centers + np.random.randn(5, 2) * 0.15

ax.scatter(img_pts[:, 0], img_pts[:, 1], s=150, c='#E74C3C', marker='s', 
           label='Image embeddings', zorder=5, edgecolors='white', linewidths=1.5)
ax.scatter(txt_pts[:, 0], txt_pts[:, 1], s=150, c='#3498DB', marker='o', 
           label='Text embeddings', zorder=5, edgecolors='white', linewidths=1.5)

for i, label in enumerate(labels):
    ax.annotate(f'img:{label}', img_pts[i], fontsize=9, 
                xytext=(5, 5), textcoords='offset points')
    ax.annotate(f'txt:{label}', txt_pts[i], fontsize=9, 
                xytext=(5, 5), textcoords='offset points')
    # Draw line connecting matched pair
    ax.plot([img_pts[i, 0], txt_pts[i, 0]], [img_pts[i, 1], txt_pts[i, 1]],
            '--', color='#2ECC71', alpha=0.7, linewidth=2)

ax.legend(fontsize=11)
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')

plt.tight_layout()
plt.savefig('../assets/alignment_problem.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. The Multimodal Model Zoo (Taxonomy)

Let's visualize the evolution and relationships between key models.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Evolution of Multimodal Models', fontsize=18, fontweight='bold', pad=20)

models = [
    # (x, y, name, year, color, type)
    (2, 8.5, 'ViT\n(2020)', '#E74C3C', 'Vision'),
    (5, 8.5, 'BERT\n(2018)', '#3498DB', 'Text'),
    (3.5, 7, 'CLIP\n(2021)', '#9B59B6', 'V+L'),
    (6.5, 7, 'ALIGN\n(2021)', '#9B59B6', 'V+L'),
    (1.5, 5.5, 'BLIP\n(2022)', '#F39C12', 'V+L'),
    (4.5, 5.5, 'Flamingo\n(2022)', '#F39C12', 'V+L'),
    (7.5, 5.5, 'BEiT-3\n(2022)', '#F39C12', 'V+L'),
    (3, 4, 'BLIP-2\n(2023)', '#2ECC71', 'V+L'),
    (6, 4, 'LLaVA\n(2023)', '#2ECC71', 'V+L'),
    (9, 4, 'InstructBLIP\n(2023)', '#2ECC71', 'V+L'),
    (4.5, 2.5, 'LLaVA-1.5\n(2024)', '#1ABC9C', 'V+L'),
    (7.5, 2.5, 'GPT-4V\n(2023)', '#1ABC9C', 'V+L'),
    
    (11, 8.5, 'Whisper\n(2022)', '#2ECC71', 'Audio'),
    (11, 7, 'ImageBind\n(2023)', '#E74C3C', 'Multi'),
    (13, 5.5, 'Gemini\n(2023)', '#1ABC9C', 'Multi'),
    (11, 4, 'Any-to-Any\n(2024)', '#34495E', 'Multi'),
]

for x, y, name, color, _ in models:
    draw_architecture_block(ax, x, y, 2.2, 0.8, name, color, fontsize=9)

# Key arrows showing evolution
connections = [
    (2, 8.0, 3.5, 7.5),
    (5, 8.0, 3.5, 7.5),
    (3.5, 6.5, 1.5, 6.0),
    (3.5, 6.5, 4.5, 6.0),
    (1.5, 5.0, 3, 4.5),
    (4.5, 5.0, 6, 4.5),
    (6, 3.5, 4.5, 3.0),
    (6, 3.5, 7.5, 3.0),
]
for x1, y1, x2, y2 in connections:
    draw_arrow(ax, (x1, y1), (x2, y2), color='gray')

# Legend
legend_items = [
    ('Vision Only', '#E74C3C'), ('Text Only', '#3498DB'),
    ('Vision+Language', '#9B59B6'), ('Multi-modal', '#1ABC9C')
]
for i, (label, color) in enumerate(legend_items):
    ax.add_patch(FancyBboxPatch((12.5, 2 - i*0.5), 0.3, 0.3, 
                                facecolor=color, alpha=0.8))
    ax.text(13, 2.15 - i*0.5, label, fontsize=10, va='center')

plt.tight_layout()
plt.savefig('../assets/model_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Hands-On: Build a Mini-CLIP Model from Scratch

Let's build the **simplest possible multimodal model** — a miniature CLIP:

```
  Image ──→ CNN Encoder ──→ Projection ──→ L2 Normalize ──→ ┐
                                                              ├──→ Cosine Similarity
  Text  ──→ Transformer  ──→ Projection ──→ L2 Normalize ──→ ┘
```

The model learns: $\text{score}(I, T) = \frac{\mathbf{v}^\top \mathbf{t}}{\|\mathbf{v}\| \|\mathbf{t}\|} \cdot \frac{1}{\tau}$

### The Temperature Parameter $\tau$

CLIP scales cosine similarities by a **temperature** $\tau$ before applying softmax:

$$p_i = \frac{\exp(s_i / \tau)}{\sum_j \exp(s_j / \tau)}$$

| Temperature | Effect | Distribution Shape |
|-------------|--------|-------------------|
| **Low** $\tau = 0.01$ | Sharp peaks | Near one-hot — very confident |
| **Medium** $\tau = 0.07$ | Balanced (CLIP default) | Discriminative but not extreme |
| **High** $\tau = 1.0$ | Uniform | All similarities treated equally |

**Why $\tau = 0.07$?** Cosine similarities live in $[-1, 1]$, but softmax needs larger logit spreads to produce meaningful gradients. Dividing by 0.07 amplifies differences: $\text{sim}=0.9 \to 12.9$ vs $\text{sim}=0.1 \to 1.4$.

**Learnable temperature:** In our `MiniCLIP`, $\tau$ is a **learnable parameter** initialized to $\exp(\log(1/0.07)) = 1/0.07 \approx 14.3$ in logit space (CLIP stores `log(1/τ)` and divides logits by `exp(log_temp)`). The model learns the optimal sharpness during training.

In [ ]:
class SimpleImageEncoder(nn.Module):
    """Tiny CNN that maps an image to an embedding vector."""
    def __init__(self, embed_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.projection = nn.Linear(128, embed_dim)

    def forward(self, x):
        x = self.features(x).flatten(1)
        return self.projection(x)


class SimpleTextEncoder(nn.Module):
    """Tiny transformer that maps token IDs to an embedding vector."""
    def __init__(self, vocab_size=1000, embed_dim=128, max_len=32):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=256, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.projection = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = self.token_embed(x) + self.pos_embed(positions)
        x = self.transformer(x)
        x = x[:, 0]  # [CLS] token representation
        return self.projection(x)


class MiniCLIP(nn.Module):
    """Minimal CLIP-like model: align image and text embeddings."""
    def __init__(self, embed_dim=128):
        super().__init__()
        self.image_encoder = SimpleImageEncoder(embed_dim)
        self.text_encoder = SimpleTextEncoder(embed_dim=embed_dim)
        self.temperature = nn.Parameter(torch.ones(1) * 0.07)

    def forward(self, images, text_ids):
        img_emb = self.image_encoder(images)
        txt_emb = self.text_encoder(text_ids)

        # L2 normalize
        img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
        txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True)

        # Cosine similarity scaled by temperature
        logits = img_emb @ txt_emb.T / self.temperature
        return logits, img_emb, txt_emb


# Detect best device (GPU on Colab, CPU locally)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# Create model and inspect
model = MiniCLIP(embed_dim=128).to(device)
count_parameters(model)

In [ ]:
# Test with dummy data (move to same device as model)
batch_size = 4
images = torch.randn(batch_size, 3, 32, 32).to(device)
text_ids = torch.randint(0, 1000, (batch_size, 16)).to(device)

logits, img_emb, txt_emb = model(images, text_ids)

print(f"Image embeddings shape: {img_emb.shape}")   # [4, 128]
print(f"Text embeddings shape:  {txt_emb.shape}")    # [4, 128]
print(f"Similarity matrix shape: {logits.shape}")     # [4, 4]
print(f"Temperature τ = {model.temperature.item():.4f}")

# Visualize the similarity matrix (move to CPU for plotting)
fig = plot_attention_heatmap(
    logits.detach().cpu(),
    x_labels=[f'text_{i}' for i in range(4)],
    y_labels=[f'img_{i}' for i in range(4)],
    title='Image-Text Similarity Matrix (untrained — random)'
)
plt.savefig('../assets/similarity_untrained.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Interpretation:")
print("  • Diagonal = matching pairs (should be HIGH after training)")
print("  • Off-diagonal = non-matching pairs (should be LOW)")
print("  • Right now it's random — training with InfoNCE loss will fix this!")

### InfoNCE Loss — Preview

The similarity matrix you just saw is trained with **InfoNCE** (Noise Contrastive Estimation) loss — the heart of contrastive learning:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \log \frac{\exp(\text{sim}(\mathbf{v}_i, \mathbf{t}_i) / \tau)}{\sum_{j=1}^{N} \exp(\text{sim}(\mathbf{v}_i, \mathbf{t}_j) / \tau)}$$

**Reading the formula:**
- **Numerator:** similarity of the **correct** image-text pair $(i, i)$
- **Denominator:** similarity of image $i$ against **all** text candidates in the batch
- The loss pushes the diagonal (matching pairs) **up** and off-diagonal (non-matching) **down**

This is exactly softmax cross-entropy where each image must "pick" its correct text caption from $N$ options. With batch size 32,768 (as in original CLIP), each sample has 32,767 negative examples — that's why CLIP needs large batches!

We explore InfoNCE in full mathematical depth in **Module 03: Training Strategies**.

## Key Takeaways

1. **Multimodal = multiple input types** sharing a learned representation space
2. Each modality needs its own **encoder** (ViT for images, BERT for text, etc.)
3. All encoders output vectors in the **same dimension** $\mathbf{h} \in \mathbb{R}^D$ — this enables fusion
4. The core challenge is **alignment**: $\text{sim}(\mathbf{v}_i, \mathbf{t}_i) \gg \text{sim}(\mathbf{v}_i, \mathbf{t}_j)$ for matching pair $i$
5. **Contrastive learning** (CLIP-style InfoNCE) is the most popular alignment method
6. Modern models (LLaVA, GPT-4V) extend this by connecting vision encoders to LLMs

---

## What's Next

In the next notebook, we'll dive deep into the **encoders** — building a Vision Transformer (ViT) and Text Encoder from scratch, with full math.

👉 **[02_modality_encoders.ipynb](./02_modality_encoders.ipynb)** — Build ViT and Text Encoder from raw PyTorch